In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.manifold import TSNE
import umap
from dtaidistance import dtw  # Import DTW library

# Step 2: Load or Simulate Data
# Assuming `latent_space` is your input data with shape (n_samples, n_frames, n_features)
latent_space = np.random.rand(32, 120, 256)  # Replace with your actual data

# Flatten the data to (n_samples, n_frames * n_features)
flattened_data = latent_space.reshape(latent_space.shape[0], -1)

# Step 3: Dimensionality Reduction with explained variance threshold for PCA, and fixed dimensionality for t-SNE and UMAP
def reduce_dimensionality(data, method="PCA", variance_threshold=0.90, n_components=2):
    if method == "PCA":
        pca = PCA()
        pca.fit(data)
        explained_variance = np.cumsum(pca.explained_variance_ratio_)
        # Find the number of components that explain the desired variance
        n_components_pca = np.argmax(explained_variance >= variance_threshold) + 1
        print(f"{method}: Number of components chosen: {n_components_pca} for {variance_threshold * 100}% variance")
        pca = PCA(n_components=n_components_pca)
        reduced_data = pca.fit_transform(data)
        print(f"{method}: Explained variance: {sum(pca.explained_variance_ratio_):.2f}")
    
    elif method == "t-SNE":
        reduced_data = TSNE(n_components=n_components).fit_transform(data)
        print(f"{method} done with {n_components} components.")
    
    elif method == "UMAP":
        reducer = umap.UMAP(n_components=n_components)
        reduced_data = reducer.fit_transform(data)
        print(f"{method} done with {n_components} components.")
    
    return reduced_data

# Step 4: Clustering Techniques
def apply_clustering(data, method="kmeans", n_clusters=5):
    if method == "kmeans":
        clustering_model = KMeans(n_clusters=n_clusters)
    elif method == "gmm":
        clustering_model = GaussianMixture(n_components=n_clusters)
    elif method == "dbscan":
        clustering_model = DBSCAN(eps=0.5, min_samples=5)
    
    clusters = clustering_model.fit_predict(data)
    return clusters

# Step 5: Evaluate Clustering
def evaluate_clustering(data, clusters):
    score = silhouette_score(data, clusters)
    print(f"Silhouette Score: {score:.2f}")

# Step 6: Similarity Metrics including DTW
def compute_similarity(data, metric="cosine"):
    if metric == "cosine":
        similarities = pairwise_distances(data, metric='cosine')
    elif metric == "euclidean":
        similarities = pairwise_distances(data, metric='euclidean')
    elif metric == "mahalanobis":
        cov_matrix = np.cov(data.T)
        inv_cov_matrix = np.linalg.inv(cov_matrix)
        similarities = pairwise_distances(data, metric='mahalanobis', VI=inv_cov_matrix)
    elif metric == "dtw":
        # DTW is typically calculated on time-series data. Here, we loop over each sequence.
        n = data.shape[0]
        dtw_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                distance = dtw.distance(data[i], data[j])  # Compute DTW distance between sequences
                dtw_matrix[i, j] = dtw_matrix[j, i] = distance
        similarities = dtw_matrix
        print("DTW similarity matrix calculated.")
    
    return similarities

# Step 7: Visualization
def visualize_clusters(reduced_data, clusters, title="Clustering Visualization"):
    plt.figure(figsize=(8,6))
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=clusters, cmap='viridis', s=100, edgecolors='k')
    plt.title(title)
    plt.grid(True)
    plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.show()

# Step 8: Full Process Comparison
methods_dim_reduction = ["PCA", "t-SNE", "UMAP"]
clustering_methods = ["kmeans", "gmm", "dbscan"]
similarity_metrics = ["cosine", "euclidean", "mahalanobis", "dtw"]  # Added DTW

# PCA with 90% variance explained, and t-SNE/UMAP with 2 components
for dim_method in methods_dim_reduction:
    if dim_method == "PCA":
        reduced_data = reduce_dimensionality(flattened_data, method=dim_method, variance_threshold=0.90)
    else:
        reduced_data = reduce_dimensionality(flattened_data, method=dim_method, n_components=2)
    
    for cluster_method in clustering_methods:
        print(f"Applying {cluster_method} clustering on {dim_method} reduced data")
        clusters = apply_clustering(reduced_data, method=cluster_method)
        evaluate_clustering(reduced_data, clusters)
        visualize_clusters(reduced_data, clusters, title=f"{cluster_method} on {dim_method} reduced data")

    # Apply similarity metrics including DTW
    for sim_metric in similarity_metrics:
        print(f"Computing {sim_metric} similarity on {dim_method} reduced data")
        similarities = compute_similarity(reduced_data, metric=sim_metric)
        print(f"Similarity matrix using {sim_metric}:\n", similarities)

# Note: Adjust the clustering parameters (e.g., n_clusters, DBSCAN eps) as necessary.
